# Validate before totaling

QuickCart is our fictional online shop. Its support staff receive made-up refund amounts, and a total is only useful if bad inputs were rejected first. Here, amounts are whole-number teaching units, not real money or a full refund policy.

> **Your goal:** Reject bad amounts before you calculate a total you can trust.<br>
> **You need:** Basic functions, lists and dictionaries.<br>
> **Effort:** 15 minutes reading and 30 minutes practice (estimates).<br>
> **You will produce:** A validated total, evidence of a failure and its fix, and checks for your own cap rule.

## Before executing

Work in your own copy, in any notebook editor that runs Python 3.11 (the version these exercises were checked with). You need no extra packages, no keys and no paid API. Running it in Colab has not been tested. Keep your results outside the course files.

Run the cells from top to bottom. Text cells explain the task; code cells do the work. An `assert` checks that something is true and raises `AssertionError` if it is not; when it passes, it usually prints nothing. Restart the kernel before your final check so leftover variables cannot hide a missed step. The notebook starts with all outputs cleared.

**Your route:** predict → read the worked example → test that bad input is rejected → break it and fix it → write your own cap rule. That last function is deliberately unfinished: defining it is not the same as calling and testing it.

[Previous: bridge route](README.md) · [Concept explanation](../../study-guides/python-engineering-bridge/README.md)

## Predict the input contract

Before you run anything, predict what should happen in each case:

| Input | Your prediction |
|---|---|
| Two records with amounts 10 and 20 | A total, or a rejection? |
| An empty list | A total, or a rejection? |
| One record with amount `True` | Is that a valid whole-number amount? |
| A negative amount, the text `"10"`, or a missing field | Convert it, ignore it or reject it? |

**The rules (the "contract"):** each record must be a dictionary with an `amount` that is a whole number of zero or more, and never `True` or `False`. Other fields are ignored in this small exercise; rejecting unknown fields and limiting batch size come later. The caller passes in a list of records. Never quietly convert a bad amount or skip a record that fails.

Think of validation as a gate in front of the calculation. It cannot tell you whether a refund is authorised or whether the refund policy itself is right.

## Worked example: validate, then accumulate

Read the next function before you run it. It checks the record's shape, then the amount, then adds it to the total. `type(amount) is int` deliberately rejects `True` and `False`, because Python lets `True` act like 1 in arithmetic. Raising a `ValueError` makes a rejected input visible to the caller, instead of returning a believable zero.

For the valid example, **10 + 20 = 30 units**. The empty list totals zero, because there was nothing invalid to reject.

In [ ]:
def total_amounts(records):
    total = 0
    for record in records:
        if not isinstance(record, dict) or "amount" not in record:
            raise ValueError("Missing amount")
        amount = record["amount"]
        if type(amount) is not int or amount < 0:
            raise ValueError("Expected nonnegative integer")
        total += amount
    return total

assert total_amounts([{"amount": 10}, {"amount": 20}]) == 30
assert total_amounts([]) == 0


In [ ]:
# Compare unsafe arithmetic with the validated contract.
def unsafe_total(records):
    return sum(record["amount"] for record in records)

assert unsafe_total([{"amount": True}]) == 1  # Successful arithmetic, wrong contract.
for records in ([{"amount": True}], [{"amount": -1}], [{}], [{"amount": "10"}]):
    try:
        total_amounts(records)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid business input accepted")


## Modify, diagnose and verify

In your own copy, temporarily replace:

```python
if type(amount) is not int or amount < 0:
```

with:

```python
if amount < 0:
```

1. Predict which bad input will now get through.
2. Rerun the cell that defines the function, then the cell that tests rejections. Just editing a cell does not change the function already in memory.
3. You should see `AssertionError: Invalid business input accepted`: the first bad case is `True`, which the weaker check now lets through.
4. Put the original condition back, rerun the definition and the same tests, then restart the kernel and run the reference cells in order again.

Keep notes on the real failure and your fix. If a later case raises a different exception, check which case ran and whether the function you meant is the one in memory. Never change the tests to hide a failure.

## Independent extension: enforce an aggregate cap

Now write the unfinished function below in your copy. The cap must be a whole number of zero or more, and not `True` or `False`. Reuse the validated total; if the total is **greater than** the cap, raise `ValueError`, otherwise return the total. Checking each item and checking the overall total are two separate jobs.

Once it is written, add a code cell below it and run these checks:

```python
assert capped_total([{"amount": 10}, {"amount": 20}], 40) == 30
assert capped_total([{"amount": 10}, {"amount": 20}], 30) == 30
assert capped_total([], 0) == 0

for records, cap in [
    ([{"amount": 30}], 20),
    ([], True), ([], -1), ([], "30"), ([], 2.5),
    ([{"amount": True}], 30), ([{}], 30),
]:
    try:
        capped_total(records, cap)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid cap or records accepted")
```

These checks call your function on purpose. Until you have written it, you will see `NotImplementedError`; that is not a successful rejection. Add one edge case of your own and explain why it matters.

In [ ]:
def capped_total(records, cap):
    # TODO: validate a nonnegative integer cap (not bool); use validated totals;
    # raise ValueError when total exceeds cap, otherwise return the total.
    raise NotImplementedError("Independent extension")

# After implementation, add normal, cap-boundary, overflow and invalid-cap assertions.
# Do not substitute the worked function and claim the extension completed.


## Hints and reflection

<details>
<summary>After your attempt: open the hints one by one</summary>

**Hint 1:** check the cap on its own, using the same "not `True`/`False`, must be `int`" rule as the amounts.

**Hint 2:** call `total_amounts(records)` instead of copying or weakening its item checks.

**Hint 3:** compare the validated total with the cap; a total equal to the cap is allowed. Raise `ValueError` if it is over, otherwise return the total.

**Why:** records that are each valid can still add up to more than an overall limit, so keep both checks. Neither check confirms who the user is or authorises a real refund.

</details>

### Troubleshoot without guessing

| Symptom | Check | Fix |
|---|---|---|
| `NameError` | Did the cell that defines it run in this kernel? | Run the earlier cells in order; restart if you are unsure what is in memory. |
| An old result is still showing after an edit | Was the changed function defined again? | Run its cell, then rerun the tests that use it. |
| `NotImplementedError` | Did you call the unfinished function? | Write it before running its checks. |
| No output after the asserts | Did the cell finish without an error? | Silence usually means success; break one case on purpose to prove the check really runs. |

**Reflect:** which bad input was most likely to slip through unnoticed? What did your regression test prove? Save your first attempt, the failure, your fix and any help you used. Nothing here starts a server or costs money; stop your kernel when you finish and clear any sensitive output before sharing.

## Check understanding and finish

**Pause and decide:** a total is exactly equal to the cap. Should it pass? And what if the cap is `True`?

<details>
<summary>Compare your reasoning</summary>

A total equal to the cap passes under the stated rule. A cap of `True` is rejected, even though Python could treat it as a number: whether the cap is valid is a separate rule.

</details>

Before you call this done, keep:

- Your prediction and the real reference result from a fresh, in-order run.
- The deliberately failing check, your fix and the check passing again.
- Asserts that really call your own function, including invalid input.
- A short explanation of the boundary, and any hints or AI help you used.

**You can now:** reject bad amounts before calculating a total you can trust, and prove it with checks rather than a saved green result.

**Next:** open [the next unit](02-typing-and-validation.ipynb) if your route includes it; otherwise go to your next selected unit in the [bridge map](README.md#unit-map-and-estimated-effort). The [package checkpoint](python-bridge/README.md) is still required before the API project. Save your evidence, clear sensitive output before sharing and stop your session.